# Food Search/Record Mismatch Detection

This notebook helps identify mismatches between what a participant searched for and what they recorded in dietary recall data. It uses fuzzy matching to flag potential mismatches for manual review.

In [1]:
# Install required packages (uncomment if running on a new environment)
# !pip install pandas fuzzywuzzy python-Levenshtein

In [1]:
import pandas as pd
import re
from fuzzywuzzy import fuzz
from nltk.stem import PorterStemmer

### Preprocessing Function
Clean and standardize the text for comparison.

In [2]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace('/', ' ')
    # previously it removed the contents of the parentheses which lost information
    # it didn't work well for  direct_match = search_tokens.issubset(recorded_tokens)
    #text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'[()]', '', text)
    text = re.sub(r'[^\w\s\']', ' ', text)  # Keep only alphanumeric and spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### Mismatch Detection Function
Compares each search/record pair, flags as mismatch if both fuzzy scores are low and the search is not a substring of the record.

In [3]:
def detect_mismatches(df, search_col, record_col, threshold=0.6):
    results = df.copy()
    results['token_set_ratio'] = 0.0
    results['partial_ratio'] = 0.0
    results['direct_match'] = False
    results['potential_mismatch'] = False
    for idx, row in results.iterrows():
        search_term = str(row[search_col])
        recorded_item = str(row[record_col])
        search_clean = preprocess_text(search_term)
        record_clean = preprocess_text(recorded_item)
        #direct_match = search_term.lower() in recorded_item.lower()
        # improved direct match that checks if all tokens in search are
        # found in the tokens of the recorded items
        stemmer = PorterStemmer()
        search_tokens = {stemmer.stem(w) for w in search_clean.split()}
        recorded_tokens = {stemmer.stem(w) for w in record_clean.split()}
        direct_match = search_tokens.issubset(recorded_tokens)
        results.at[idx, 'direct_match'] = direct_match
        token_set = fuzz.token_set_ratio(search_tokens, recorded_tokens) / 100
        partial = fuzz.partial_ratio(search_tokens, recorded_tokens) / 100
        results.at[idx, 'token_set_ratio'] = token_set
        # remove partial and threshold, they don't work well
        # as they are bidirectional. Of the 2, token should work better
        # but doesn't work if all the terms in recorded are in searched - gives 100 score
        # results.at[idx, 'partial_ratio'] = partial
        # if token_set < threshold and partial < threshold and not direct_match:
        #     results.at[idx, 'potential_mismatch'] = True
        if not direct_match:
             results.at[idx, 'potential_mismatch'] = True
    return results

### Example Test
See how the algorithm works on example data.

In [18]:
# Example data
examples = pd.DataFrame({
    'search': ["satsume", "orange juice", "tea", "Cracker,satsuma,Water"],
    'recorded': ["tangerines/ mandarins/ satsumas", "orange juice, 100% fruit (e.g. Tropicana)", "tea", "Cream cracker,Tangerines / mandarins / clementines/ satsumas,Water (from tap, including filtered)"]
})
results = detect_mismatches(examples, 'search', 'recorded')
results

,search,recorded,token_set_ratio,partial_ratio,direct_match,potential_mismatch
0,satsume,tangerines/ mandarins/ satsumas,0.39,0.86,False,False
1,orange juice,"orange juice, 100% fruit (e.g. Tropicana)",1.00,1.00,True,False
2,tea,tea,1.00,1.00,True,False
3,"Cracker,satsuma,Water","Cream cracker,Tangerines / mandarins / clement...",0.76,0.71,True,False


In [10]:
preprocess_text("Cracker,satsuma,Water")

'cracker satsuma water'

In [9]:
preprocess_text("Cream cracker,Tangerines / mandarins / clementines/ satsumas,Water (from tap, including filtered)")

'cream cracker tangerines mandarins clementines satsumas water'

In [9]:
# search_term = preprocess_text("Cracker,satsuma,Water")
# recorded_term = preprocess_text("Cream cracker,Tangerines / mandarins / clementines/ satsumas,Water (from tap, including filtered)")
search_term = preprocess_text("Fries,Chicken nuggets,Ketchup,Strawberry Milkshake")
recorded_term = preprocess_text("Takeaway chips, fries (McDonalds Only),Chicken nuggets/pieces, from takeaway e.g McNuggets,Tomato ketchup,Milkshake / Milk drink, not chocolate (e.g. strawberry, banana)")

stemmer = PorterStemmer()

search_tokens = {stemmer.stem(w) for w in search_term.split()}
recorded_tokens = {stemmer.stem(w) for w in recorded_term.split()}

direct_match = search_tokens.issubset(recorded_tokens)
score = fuzz.token_set_ratio(search_tokens, recorded_tokens)

In [10]:
direct_match, search_term, recorded_term,search_tokens, recorded_tokens, score

(True,
 'fries chicken nuggets ketchup strawberry milkshake',
 'takeaway chips fries mcdonalds only chicken nuggets pieces from takeaway e g mcnuggets tomato ketchup milkshake milk drink not chocolate e g strawberry banana',
 {'chicken', 'fri', 'ketchup', 'milkshak', 'nugget', 'strawberri'},
 {'banana',
  'chicken',
  'chip',
  'chocol',
  'drink',
  'e',
  'fri',
  'from',
  'g',
  'ketchup',
  'mcdonald',
  'mcnugget',
  'milk',
  'milkshak',
  'not',
  'nugget',
  'onli',
  'piec',
  'strawberri',
  'takeaway',
  'tomato'},
 100)

### To Use With Your Data
Replace the file names and column names as appropriate.

In [5]:
# Uncomment and edit to use with your data
# df = pd.read_csv('Y12-15_intake24_grouped_output.csv')
# df = pd.read_csv('Intake24_Recall_Items_MF1L - for Tom - grouped.csv')
df = pd.read_csv('Y17_MF1L_withAFP_for_Tom_grouped.csv')
# results = detect_mismatches(df, 'SearchTerm', 'DescriptionEn', threshold=0.95)
results = detect_mismatches(df, 'SearchTerm', 'NameEn', threshold=0.95)
# results.to_csv('simple_matches_fuzzy_2.csv', index=False)
results.to_csv('Y17_MF1L_withAFP_for_Tom_grouped_mismatches.csv', index=False)
print(f"{sum(results['potential_mismatch'])} potential mismatches flagged out of {len(results)} rows.")

19576 potential mismatches flagged out of 47884 rows.
